# Challenge Four: Programming an Agent Workflow

**Goal:** Demonstrate the ability to program a complex process using ADK workflow agents.

**Requirements covered in this notebook:**
1. An agent that answers questions but verifies and refines the answer before returning it.
2. A workflow with multiple agents that answer, verify (critique), and refine the response.

**Architecture:**
- `search_agent` -- researches the question with Google Search and writes a draft answer.
- `critique_agent` -- reviews the draft and lists concrete suggestions for improving it.
- `refine_agent` -- rewrites the draft, incorporating the critique, into the final answer.
- `answer_team` -- a `SequentialAgent` that runs the three specialists above in order,
  passing each one's output to the next via session state.
- `greeter` (root agent) -- the entry point; delegates every question straight to
  `answer_team`.




In [1]:
# 1. Install dependencies
!pip install --upgrade --quiet google-adk google-cloud-aiplatform


In [2]:
# 2. Imports and configuration
import os
import logging
from typing import Optional

from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search

# --- Configuration ---
# GOOGLE_CLOUD_PROJECT / GOOGLE_CLOUD_LOCATION: your Cloud Skills Boost lab project ID
#   (shown on your Qwiklabs lab page) and a Vertex AI region.

MODEL_GEMINI_FLASH = "gemini-2.5-flash"

GOOGLE_CLOUD_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "qwiklabs-gcp-00-6263dfcac21a")
GOOGLE_CLOUD_LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

import vertexai
vertexai.init(project=GOOGLE_CLOUD_PROJECT, location=GOOGLE_CLOUD_LOCATION)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("agent_workflow")


## The three-stage answer pipeline

Each agent below has an `output_key`, which tells ADK to save that agent's
final text response into session state under that key. Later agents in the
same `SequentialAgent` reference earlier agents' output directly in their
instruction text using `{state_key}` placeholders -- ADK automatically
substitutes session state into plain-string instructions before calling the
model, so no extra wiring is needed.


In [3]:
# 3. Search agent: researches the question and drafts an initial answer
SEARCH_AGENT_INSTRUCTIONS = """
You are a research assistant. Use Google Search to find accurate, up-to-date
information that answers the user's question.

Write a clear, well-supported draft answer based on what you find. Keep it
factual and cite specifics (names, numbers, dates) where relevant. This is a
first draft -- a reviewer will critique it next, so it does not need to be
perfect, but it should be accurate and directly address the question.
"""

search_agent = Agent(
    name="search_agent",
    model=MODEL_GEMINI_FLASH,
    description="Researches the user's question with Google Search and writes a draft answer.",
    instruction=SEARCH_AGENT_INSTRUCTIONS,
    tools=[google_search],
    output_key="draft_answer",
    # The built-in google_search tool cannot be combined with any other
    # function-declaration tool (e.g. an auto-injected transfer tool) in the
    # same model call. This agent has no siblings/parent that need it to
    # transfer control, so disable that mechanism defensively.
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
)


In [4]:
# 4. Critique agent: reviews the draft and suggests improvements
CRITIQUE_AGENT_INSTRUCTIONS = """
You are a careful editorial reviewer. You will be shown a draft answer to a
user's question:

--- DRAFT ANSWER ---
{draft_answer}
--- END DRAFT ANSWER ---

Review it for accuracy, completeness, and clarity. Write a short, specific,
actionable list of suggestions for how to improve it (e.g. missing details,
unclear phrasing, unsupported claims). If the draft is already excellent and
needs no changes, say so explicitly and clearly (e.g. "No changes needed.").

Only output the review notes -- do not rewrite the answer yourself.
"""

critique_agent = Agent(
    name="critique_agent",
    model=MODEL_GEMINI_FLASH,
    description="Reviews the draft answer and suggests concrete improvements.",
    instruction=CRITIQUE_AGENT_INSTRUCTIONS,
    output_key="critique_notes",
)


In [5]:
# 5. Refine agent: rewrites the draft using the critique
REFINE_AGENT_INSTRUCTIONS = """
You will be shown a draft answer and a reviewer's critique of it:

--- DRAFT ANSWER ---
{draft_answer}
--- END DRAFT ANSWER ---

--- REVIEWER NOTES ---
{critique_notes}
--- END REVIEWER NOTES ---

Rewrite the draft answer, applying the reviewer's suggestions (if the notes
say no changes are needed, just clean up the draft's wording). Output only
the final, polished answer to the user's original question -- no
meta-commentary about the review process.
"""

refine_agent = Agent(
    name="refine_agent",
    model=MODEL_GEMINI_FLASH,
    description="Rewrites the draft answer to incorporate the reviewer's suggested improvements.",
    instruction=REFINE_AGENT_INSTRUCTIONS,
    output_key="final_answer",
)


## The workflow and the entry point

`answer_team` is a `SequentialAgent`: it always runs `search_agent`, then
`critique_agent`, then `refine_agent`, in that fixed order -- exactly matching
the answer -> verify -> refine requirement. `greeter` is the root agent the
user actually talks to; its only job is to hand every question straight to
`answer_team`.


In [6]:
# 6. Sequential workflow: search -> critique -> refine
answer_team = SequentialAgent(
    name="answer_team",
    description=(
        "Answers a question by researching a draft, critiquing it, and "
        "refining it into a final response."
    ),
    sub_agents=[search_agent, critique_agent, refine_agent],
)


/tmp/ipykernel_30455/619282884.py:2: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_team = SequentialAgent(


In [7]:
# 7. Root agent: the entry point, delegates every question to the answer team
GREETER_INSTRUCTIONS = """
You are the friendly entry point for a question-answering assistant. You do
not answer questions yourself. As soon as the user asks a question, delegate
it to the `answer_team` sub-agent, which will research, critique, and refine
a high-quality response before it is shown to the user.
"""

greeter = Agent(
    name="greeter",
    model=MODEL_GEMINI_FLASH,
    description="Entry point that greets the user and delegates their question to the answer team.",
    instruction=GREETER_INSTRUCTIONS,
    sub_agents=[answer_team],
)


## Running the workflow

Same event-trace approach as Challenge Three: print every streamed event
(with its `author`) so the notebook output makes the hand-offs visible --
`greeter` transferring to `answer_team`, then `search_agent`,
`critique_agent`, and `refine_agent` each doing their part in turn.


In [8]:
# 8. Helper to run a query and print every event (proves the workflow ran end to end)
from vertexai.preview import reasoning_engines
from IPython.display import Markdown, display

def describe_event(event: dict) -> None:
    """
    Print a one-line-per-part summary of a single streamed event: which agent
    authored it, and whether it is text, a tool call, or a tool result.

    Args:
        event (dict): One event dict from AdkApp.stream_query().
    """
    author = event.get("author", "?")
    content = event.get("content") or {}
    for part in content.get("parts", []) or []:
        if part.get("text"):
            text = part.get("text", "").strip()[:200]
            print(f"  [{author}] TEXT  \u00bb {text}")
        elif part.get("function_call"):
            fc = part["function_call"]
            fc_name = fc.get("name")
            fc_args = fc.get("args")
            print(f"  [{author}] CALL  \u00bb {fc_name}({fc_args})")
        elif part.get("function_response"):
            fr = part["function_response"]
            fr_name = fr.get("name")
            print(f"  [{author}] RESULT \u00bb from {fr_name}")


def ask_agent_verbose(agent: Agent, question: str, user_id: str = "test-user-id") -> Optional[str]:
    """
    Host the given agent in an AdkApp, create a session, query it once, and
    print every event along the way before returning the final response text.

    Args:
        agent (Agent): The ADK agent to run (typically the root agent).
        question (str): The natural-language question/prompt to send.
        user_id (str): An identifier for the querying user/session owner.

    Returns:
        Optional[str]: The text of the agent\'s final response, or None on error.
    """
    app = reasoning_engines.AdkApp(agent=agent)
    session = app.create_session(user_id=user_id)
    session_id = session["id"] if isinstance(session, dict) else session.id

    last_event = None
    try:
        for event in app.stream_query(user_id=user_id, session_id=session_id, message=question):
            describe_event(event)
            last_event = event
    except Exception as e:
        print(f"Error while querying agent \'{agent.name}\': {e}")
        return None

    if not last_event or "content" not in last_event:
        print(f"Agent \'{agent.name}\' did not return a valid final response.")
        return None

    return last_event["content"]["parts"][0]["text"]


## Test code

A factual question, run through the whole `greeter` -> `answer_team` ->
(`search_agent` -> `critique_agent` -> `refine_agent`) pipeline. Watch the
event trace: you should see `search_agent` calling Google Search, then
`critique_agent` producing review notes, then `refine_agent` producing the
final answer -- proving all three sub-agents ran as part of the workflow.


In [9]:
# 10. Test: ask a factual question and watch it flow through the whole pipeline
question = "What is the Google Agent Development Kit (ADK) and what is it used for?"

print(f"=== Query: {question} ===")
final_response = ask_agent_verbose(greeter, question)
print()
display(Markdown(final_response or "*(no response)*"))


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


=== Query: What is the Google Agent Development Kit (ADK) and what is it used for? ===


/usr/local/lib/python3.12/dist-packages/google/adk/tools/transfer_to_agent_tool.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  function_decl = super()._get_declaration()


  [greeter] CALL  » transfer_to_agent({'agent_name': 'answer_team'})
  [greeter] RESULT » from transfer_to_agent
  [search_agent] TEXT  » The Google Agent Development Kit (ADK) is an open-source framework provided by Google designed to help developers build, evaluate, and deploy smart AI agents, particularly focusing on multi-agent syst
  [critique_agent] TEXT  » The draft answer is very comprehensive, accurate, and well-structured. It clearly defines ADK and effectively outlines its key uses and benefits.

No changes needed.
  [refine_agent] TEXT  » The Google Agent Development Kit (ADK) is an open-source framework provided by Google designed to help developers build, evaluate, and deploy smart AI agents, particularly focusing on multi-agent syst



The Google Agent Development Kit (ADK) is an open-source framework provided by Google designed to help developers build, evaluate, and deploy smart AI agents, particularly focusing on multi-agent systems. It shifts the paradigm from treating large language models as simple request-response systems to building AI-powered applications with a more structured, software development approach.

The ADK is used for:
*   **Building Multi-Agent Systems** Unlike many existing agent frameworks that focus on a single, powerful agent, ADK is built from the ground up for systems where multiple specialized AI agents collaborate, delegate tasks, and communicate to solve complex problems. This allows for the creation of networks of smaller, purpose-built agents that work together like a coordinated team.
*   **Developing Production-Ready AI Agents** ADK provides a robust, event-driven framework for building reliable AI agents at an enterprise scale, ranging from personal AI assistants to mission-critical business workflows.
*   **Facilitating Interoperability** It runs on Google's Agent Protocol, enabling agents from different companies to interact securely. ADK also includes "Artifact," a built-in system for storing and versioning structured data, documents, audio, and video, making it suitable for multimodal AI applications.
*   **Streamlining Agent Development** ADK offers easy-to-use building blocks for agents, tools, memory, orchestration patterns, evaluation, and deployment. It treats agents as software components, tools as regular functions, and systems as composable modules, providing a code-first framework. This also means a strong developer experience, including built-in debugging tools that visualize prompts, model requests, tool calls, and state transitions.
*   **Integration with Google's Ecosystem** While flexible enough to work with any AI model, ADK is deeply integrated with Google's cloud services, including Gemini models and Vertex AI, offering direct connectors for secure access to structured data and enterprise applications.
*   **Ensuring Security** ADK addresses security challenges in multi-agent systems, such as trust and data misuse, by incorporating mechanisms to build secure, zero-trust AI agents.

ADK is available in multiple programming languages, including Python, TypeScript, Go, and Java. It is designed to provide developers with precise control over agent behavior and orchestration, offering a flexible and modular foundation for the next generation of AI applications.

## Notes

- Replace any placeholder configuration with your real Google Cloud project details
  before running (see the `vertexai.init(...)` cell used in earlier challenges).
- `output_key` is what makes the hand-off between sequential agents work: each agent's
  final text is written to session state under that key, and the next agent's plain
  string instruction automatically has those `{key}` placeholders substituted in by ADK
  before the model call -- no manual state plumbing required.
- If you want to see the raw intermediate values (the draft answer and the critique
  notes, not just the final one), print `session.state` after the run, or add print
  statements inside `describe_event` for events with an `actions.state_delta`.
- You may see a `DeprecationWarning` mentioning that `SequentialAgent` is deprecated in
  favor of a newer `Workflow` API. That's expected and harmless -- the warning itself
  notes `Workflow` can't yet be used as a sub-agent under an `LlmAgent` (which is exactly
  what `answer_team` is here), so `SequentialAgent` remains the correct choice.
